# 🧠 Fondamentaux de l'IA, du ML et du Deep Learning — Notebook Bootcamp Data Science

## Module Data Science — Deep Learning

> **Prérequis :** cours `01-DeepLearning_Fundamentals_AI_ML_DL.md`.

Ce notebook rend **concrètes** les grandes idées du cours. Chaque partie part d'une affirmation du cours
et la **vérifie par l'expérience**. Il est autonome (datasets scikit-learn, aucun CSV) et rapide à exécuter.

| Partie | Affirmation du cours testée |
|--------|------------------------------|
| **1** | Programmation classique : données + règles → réponses. ML : données + réponses → règles |
| **2** | Un neurone = somme pondérée + biais, puis activation |
| **3** | Les fonctions d'activation et leurs rôles (sigmoïde, tanh, ReLU, softmax) |
| **4** | Sans non-linéarité, un réseau profond = une seule couche linéaire |
| **5** | La propagation avant, pour un exemple puis pour tout un lot |
| **6** | La descente de gradient et l'importance du learning rate |
| **7** | Le Deep Learning apprend lui-même ses features |
| **8** | Epochs, batchs, itérations — et le surapprentissage |
| **9** | Sur des données tabulaires, le ML classique fait aussi bien (ou mieux) |
| **10** | Un modèle reproduit les biais de ses données |

> 🔗 Le notebook du **Checkpoint 9** (`02-...ipynb`) approfondit le neurone et `MLPClassifier` ;
> le notebook `03-...ipynb` détaille la forward et la backpropagation.

In [ ]:
!pip install scikit-learn -q

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time
import warnings
warnings.filterwarnings('ignore')            # masque les avertissements de convergence de scikit-learn

np.set_printoptions(precision=4, suppress=True)
rng = np.random.default_rng(42)

def sigmoid(z): return 1 / (1 + np.exp(-z))
def relu(z):    return np.maximum(0, z)

print('Environnement prêt ✅')

---
# PARTIE 1 — Programmation classique vs Machine Learning

```
PROGRAMMATION CLASSIQUE          MACHINE LEARNING
Données + Règles → Réponses      Données + Réponses → Règles (apprises)
```

Exemple : convertir des degrés Celsius en Fahrenheit.

## 1.1 Approche classique : on écrit la règle

In [ ]:
def celsius_vers_fahrenheit(c):
    return c * 1.8 + 32          # la RÈGLE, écrite par un humain qui la connaît

for c in [-10, 0, 25, 37, 100]:
    print(f'{c:>4} °C -> {celsius_vers_fahrenheit(c):6.1f} °F')

## 1.2 Approche Machine Learning : on apprend la règle à partir d'exemples

On suppose cette fois qu'on **ne connaît pas** la formule. On dispose seulement de **mesures** (un peu bruitées,
comme de vraies mesures) : des couples (°C, °F). Le modèle doit **retrouver la règle**.

In [ ]:
from sklearn.linear_model import LinearRegression

celsius = rng.uniform(-40, 100, size=50)                              # 50 mesures
fahrenheit = celsius * 1.8 + 32 + rng.normal(0, 1.5, size=50)        # « réponses » observées, bruitées

modele = LinearRegression().fit(celsius.reshape(-1, 1), fahrenheit)
print(f'Règle apprise : F = {modele.coef_[0]:.3f} × C + {modele.intercept_:.2f}')
print('Règle exacte  : F = 1.800 × C + 32.00')

plt.figure(figsize=(7, 4))
plt.scatter(celsius, fahrenheit, s=18, label='exemples (données + réponses)')
grille = np.linspace(-40, 100, 2).reshape(-1, 1)
plt.plot(grille, modele.predict(grille), color='darkorange', lw=2.5, label='règle apprise')
plt.xlabel('°C'); plt.ylabel('°F'); plt.legend(); plt.grid(alpha=.3)
plt.title('Le ML retrouve la règle à partir des exemples'); plt.show()

> 🔑 Le modèle a **retrouvé** les coefficients 1,8 et 32 sans qu'on les lui donne. C'est tout l'intérêt du ML :
> quand la règle est **inconnue** ou **trop complexe** pour être écrite (reconnaître un chat, détecter une fraude),
> on la fait apprendre à partir d'exemples.
>
> 💡 Ce modèle est en fait **un seul neurone sans activation** : $\hat{y} = w x + b$, avec $w \approx 1{,}8$ et $b \approx 32$.

---
# PARTIE 2 — Le neurone artificiel

$$z = \sum_{i=1}^{n} w_i x_i + b \qquad\qquad a = f(z)$$

In [ ]:
x = np.array([0.5, 0.3, 0.2])      # entrées (features)
w = np.array([0.4, 0.7, 0.1])      # poids (APPRIS)
b = 0.1                             # biais (APPRIS)

z = np.dot(w, x) + b               # étape 1 : somme pondérée + biais
a = sigmoid(z)                      # étape 2 : activation
print('Détail : ' + ' + '.join(f'{wi}×{xi}' for wi, xi in zip(w, x)) + f' + {b} = {z:.2f}')
print(f'z = {z:.2f}  ->  a = sigmoïde(z) = {a:.4f}')

In [ ]:
# Que se passe-t-il quand on modifie un poids ou le biais ?
essais = {
    'référence':               (w, b),
    'w₂ : 0.7 -> 2.0':         (np.array([0.4, 2.0, 0.1]), b),
    'w₂ : 0.7 -> -2.0':        (np.array([0.4, -2.0, 0.1]), b),
    'b : 0.1 -> -2.0':         (w, -2.0),
}
pd.DataFrame([{'réglage': nom, 'z': np.dot(wi, x) + bi, 'sortie a': sigmoid(np.dot(wi, x) + bi)}
              for nom, (wi, bi) in essais.items()]).round(4)

> 🔑 Un **poids** amplifie, atténue ou **inverse** l'effet d'une entrée ; le **biais** rend le neurone plus ou moins
> facile à activer. **Apprendre**, c'est trouver les bonnes valeurs de tous ces réglages.

---
# PARTIE 3 — Les fonctions d'activation

In [ ]:
z = np.linspace(-5, 5, 400)
fig, axes = plt.subplots(1, 3, figsize=(14, 3.6))
for ax, (nom, f, plage) in zip(axes, [('Sigmoïde', sigmoid, '[0, 1]'),
                                      ('Tanh', np.tanh, '[-1, 1]'),
                                      ('ReLU', relu, '[0, +∞[')]):
    ax.plot(z, f(z), lw=2.5); ax.axhline(0, color='gray', lw=.8); ax.axvline(0, color='gray', lw=.8)
    ax.set_title(f'{nom} — sortie dans {plage}'); ax.set_xlabel('z'); ax.grid(alpha=.3)
plt.tight_layout(); plt.show()

## 3.1 Softmax : des scores aux probabilités

En classification multiclasse, la couche de sortie produit un **score** par classe ; la **softmax** les transforme
en probabilités positives qui **somment à 1**.

$$\text{softmax}(z)_k = \frac{e^{z_k}}{\sum_j e^{z_j}}$$

In [ ]:
def softmax(z):
    e = np.exp(z - z.max())          # on soustrait le max : même résultat, mais évite les débordements numériques
    return e / e.sum()

scores = np.array([2.0, 1.0, 0.1])   # scores bruts pour 3 classes : chat, chien, oiseau
probas = softmax(scores)
for classe, s, p in zip(['chat', 'chien', 'oiseau'], scores, probas):
    print(f'{classe:<7} score {s:4.1f} -> probabilité {p:.3f}')
print('Somme des probabilités :', probas.sum())

> 🔑 Les réflexes : **ReLU** dans les couches cachées, **sigmoïde** pour une sortie binaire, **softmax** pour une sortie
> multiclasse.

---
# PARTIE 4 — Pourquoi la non-linéarité est indispensable

Le cours affirme : *sans activation non linéaire, un réseau profond se réduit à une seule couche linéaire*.
Vérifions-le de deux façons.

## 4.1 La preuve par le calcul

Deux couches **sans activation** : $W_2(W_1 x + b_1) + b_2 = (W_2 W_1)\,x + (W_2 b_1 + b_2)$ — c'est **une seule**
couche linéaire, de poids $W = W_2 W_1$ et de biais $b = W_2 b_1 + b_2$.

In [ ]:
W1, b1 = rng.normal(size=(4, 2)), rng.normal(size=4)     # couche 1 : 2 entrées -> 4 neurones
W2, b2 = rng.normal(size=(1, 4)), rng.normal(size=1)     # couche 2 : 4 -> 1

x_test = rng.normal(size=(5, 2))                         # 5 exemples
deux_couches = (x_test @ W1.T + b1) @ W2.T + b2          # réseau à 2 couches, SANS activation
W, b = W2 @ W1, W2 @ b1 + b2                             # la couche unique équivalente
une_couche = x_test @ W.T + b

print('Deux couches linéaires :', deux_couches.ravel())
print('Une seule couche       :', une_couche.ravel())
print('Identiques ?', np.allclose(deux_couches, une_couche))

## 4.2 La preuve par l'expérience : deux cercles imbriqués

Aucune droite ne sépare ces deux classes. Entraînons le **même** réseau (2 couches cachées de 16 neurones)
avec une activation **identité** (linéaire), puis avec **ReLU**.

In [ ]:
from sklearn.datasets import make_circles
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier

Xc, yc = make_circles(n_samples=1000, noise=0.08, factor=0.5, random_state=42)
Xc_tr, Xc_te, yc_tr, yc_te = train_test_split(Xc, yc, test_size=0.25, random_state=42, stratify=yc)

reseaux = {}
for activation in ['identity', 'relu']:
    reseaux[activation] = MLPClassifier(hidden_layer_sizes=(16, 16), activation=activation,
                                        max_iter=2000, random_state=42).fit(Xc_tr, yc_tr)
    print(f'activation = {activation:<8} -> accuracy test : {reseaux[activation].score(Xc_te, yc_te):.3f}')

def tracer_frontiere(ax, modele, X, y, titre):
    xx, yy = np.meshgrid(np.linspace(X[:, 0].min() - .3, X[:, 0].max() + .3, 300),
                         np.linspace(X[:, 1].min() - .3, X[:, 1].max() + .3, 300))
    proba = modele.predict_proba(np.c_[xx.ravel(), yy.ravel()])[:, 1].reshape(xx.shape)
    ax.contourf(xx, yy, proba, levels=20, cmap='RdBu_r', alpha=.6)
    ax.scatter(X[:, 0], X[:, 1], c=y, cmap='RdBu_r', s=10, edgecolor='k', linewidth=.2)
    ax.set_title(titre); ax.set_xticks([]); ax.set_yticks([])

fig, axes = plt.subplots(1, 2, figsize=(11, 5))
for ax, act in zip(axes, ['identity', 'relu']):
    tracer_frontiere(ax, reseaux[act], Xc_te, yc_te,
                     f"activation '{act}' — accuracy {reseaux[act].score(Xc_te, yc_te):.0%}")
plt.show()

> 🧐 Avec l'activation **identité**, le réseau — pourtant profond — ne peut tracer qu'une **droite** : il ne fait pas mieux
> que le hasard (40 % lors de notre exécution). Avec **ReLU**, il dessine une frontière **fermée** autour du cercle intérieur
> et sépare les classes presque parfaitement (100 %).
> C'est la non-linéarité qui rend la profondeur utile.

---
# PARTIE 5 — La propagation avant (forward)

Le réseau du cours : **3 entrées → 2 neurones cachés (ReLU) → 1 sortie (sigmoïde)**. Chaque couche calcule
$a = f(Wx + b)$.

In [ ]:
x  = np.array([0.5, 0.3, 0.2])
W1 = np.array([[0.4, 0.7, 0.1],
               [0.2, 0.5, 0.6]])        # (2 neurones, 3 entrées)
b1 = np.array([0.1, 0.2])
W2 = np.array([[0.3, 0.9]])             # (1 neurone, 2 entrées)
b2 = np.array([0.05])

z1 = W1 @ x + b1;   a1 = relu(z1)      # couche cachée
z2 = W2 @ a1 + b2;  y_hat = sigmoid(z2)  # sortie
print('Couche cachée : z1 =', z1, '-> a1 =', a1)
print('Sortie        : z2 =', z2, '-> ŷ =', y_hat)

## 5.1 Tout un lot d'exemples en une seule opération

En pratique, on fait passer **plusieurs exemples à la fois** (un *batch*) : on les empile en lignes d'une matrice $X$,
et la même formule s'écrit $A = f(XW^\top + b)$. C'est ce qui rend les GPU si efficaces.

In [ ]:
X_lot = rng.uniform(0, 1, size=(4, 3))          # 4 exemples, 3 features chacun
A1 = relu(X_lot @ W1.T + b1)                     # (4, 2)
Y_hat = sigmoid(A1 @ W2.T + b2)                  # (4, 1)
print('Formes :', X_lot.shape, '->', A1.shape, '->', Y_hat.shape)
print('Prédictions du lot :', Y_hat.ravel())

# Vérification : le calcul en lot donne la même chose qu'une boucle exemple par exemple
boucle = [sigmoid(W2 @ relu(W1 @ xi + b1) + b2)[0] for xi in X_lot]
print('Identique à la boucle ?', np.allclose(Y_hat.ravel(), boucle))

---
# PARTIE 6 — La descente de gradient

> 💡 **La montagne dans le brouillard** : on ne voit pas le fond de la vallée, mais on sent la **pente** (le gradient)
> et on fait un pas vers le bas : $w \leftarrow w - \eta \cdot \dfrac{dL}{dw}$.

Prenons une loss très simple, $L(w) = (w - 3)^2 + 1$, dont le minimum est en $w = 3$, et partons de $w = -2$.

In [ ]:
def loss(w):     return (w - 3) ** 2 + 1
def gradient(w): return 2 * (w - 3)

def descente(eta, w0=-2.0, n_pas=12):
    ws = [w0]
    for _ in range(n_pas):
        ws.append(ws[-1] - eta * gradient(ws[-1]))   # un pas dans le sens qui fait baisser la loss
    return np.array(ws)

etas = {0.05: 'trop petit', 0.3: 'adapté', 0.9: 'grand : oscille', 1.05: 'trop grand : diverge'}
grille = np.linspace(-6, 12, 300)
fig, axes = plt.subplots(1, 4, figsize=(17, 3.8), sharey=True)
for ax, (eta, verdict) in zip(axes, etas.items()):
    ws = descente(eta)
    ax.plot(grille, loss(grille), color='lightgray', lw=2)
    ax.plot(ws, loss(ws), 'o-', color='darkorange', ms=4)
    ax.set_title(f'η = {eta} ({verdict})\nw final = {ws[-1]:.2f}', fontsize=10)
    ax.set_xlabel('w'); ax.set_ylim(0, 90); ax.grid(alpha=.3)
axes[0].set_ylabel('Loss')
plt.tight_layout(); plt.show()

> 🔑 Avec un **petit** learning rate, on descend lentement ; avec un learning rate **adapté**, on atteint vite le minimum ;
> **trop grand**, on rebondit d'un versant à l'autre, et **au-delà d'un seuil**, on s'éloigne du minimum à chaque pas :
> l'entraînement **diverge**. C'est pourquoi le learning rate est l'hyperparamètre le plus sensible.

---
# PARTIE 7 — Le Deep Learning apprend lui-même ses features

En ML classique, un expert conçoit les features (« contours », « textures »…). Un réseau de neurones les **apprend**.
Entraînons un petit réseau sur des images de chiffres manuscrits (8 × 8 pixels), puis regardons ce que sa couche cachée
a appris.

In [ ]:
from sklearn.datasets import load_digits

digits = load_digits()
Xd = digits.data / 16.0                    # 1 797 images de 64 pixels, valeurs dans [0, 1]
yd = digits.target
Xd_tr, Xd_te, yd_tr, yd_te = train_test_split(Xd, yd, test_size=0.25, random_state=42, stratify=yd)

fig, axes = plt.subplots(1, 10, figsize=(14, 1.8))
for k, ax in enumerate(axes):
    ax.imshow(digits.images[np.where(yd == k)[0][0]], cmap='gray_r'); ax.set_title(str(k)); ax.axis('off')
plt.show()

mlp_digits = MLPClassifier(hidden_layer_sizes=(16,), max_iter=1000, random_state=42).fit(Xd_tr, yd_tr)
print(f'Accuracy test : {mlp_digits.score(Xd_te, yd_te):.3f}  (avec seulement 16 neurones cachés)')

In [ ]:
# Chaque neurone caché a 64 poids, un par pixel : on peut les afficher comme une image 8×8
poids = mlp_digits.coefs_[0]                      # (64 pixels, 16 neurones)
m = np.abs(poids).max()
fig, axes = plt.subplots(2, 8, figsize=(14, 3.8))
for k, ax in enumerate(axes.ravel()):
    ax.imshow(poids[:, k].reshape(8, 8), cmap='RdBu_r', vmin=-m, vmax=m)
    ax.set_title(f'neurone {k}', fontsize=9); ax.axis('off')
fig.suptitle('Ce que « regarde » chaque neurone caché (rouge : pixel qui l\'active, bleu : pixel qui l\'inhibe)')
plt.tight_layout(); plt.show()

> 🧐 Personne n'a dit au réseau quoi chercher : chaque neurone a **inventé** son propre détecteur, une combinaison de pixels
> qui l'active (rouge) ou l'inhibe (bleu). Ces motifs ne sont pas forcément lisibles à l'œil, mais ils suffisent à
> reconnaître 97 % des chiffres. Certains neurones peuvent rester gris uniforme : ce sont des neurones ReLU « morts »,
> qui ne s'activent jamais et n'apprennent plus. Dans un réseau profond, les couches suivantes combinent ces détecteurs
> en motifs de plus en plus abstraits — c'est la **hiérarchie de features** décrite dans le cours (et visible dans les
> CNN, notebook `04-...ipynb`).

---
# PARTIE 8 — Epochs, batchs, itérations… et surapprentissage

| Terme | Définition |
|-------|------------|
| **Epoch** | un passage complet sur tout le jeu d'entraînement |
| **Batch** | un sous-groupe d'exemples traités ensemble avant une mise à jour |
| **Itération** | une mise à jour des poids (un batch) |

In [ ]:
n_train = len(Xd_tr)
for batch in [1, 32, 200, n_train]:
    iterations = int(np.ceil(n_train / batch))
    print(f'batch = {batch:>5} -> {iterations:>5} itérations par epoch  | 20 epochs = {20 * iterations:>6} mises à jour')

## 8.1 Trop d'epochs : le réseau mémorise

On entraîne un **grand** réseau (256 neurones) sur **peu** d'images (200), epoch par epoch, en mesurant la loss
sur l'entraînement **et** sur un jeu de validation.

In [ ]:
from sklearn.metrics import log_loss

X_petit, y_petit = Xd_tr[:200], yd_tr[:200]
X_val, y_val = Xd_te, yd_te

reseau = MLPClassifier(hidden_layer_sizes=(256,), learning_rate_init=0.01, random_state=42)
historique = []
for epoch in range(1, 151):
    reseau.partial_fit(X_petit, y_petit, classes=np.arange(10))     # une epoch de plus
    historique.append({'epoch': epoch,
                       'loss train': log_loss(y_petit, reseau.predict_proba(X_petit), labels=np.arange(10)),
                       'loss validation': log_loss(y_val, reseau.predict_proba(X_val), labels=np.arange(10)),
                       'acc train': reseau.score(X_petit, y_petit),
                       'acc validation': reseau.score(X_val, y_val)})
h = pd.DataFrame(historique).set_index('epoch')

meilleure = h['loss validation'].idxmin()
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
h[['loss train', 'loss validation']].plot(ax=axes[0], lw=2)
axes[0].axvline(meilleure, color='green', ls='--', label=f'meilleure validation (epoch {meilleure})')
axes[0].set_ylabel('loss'); axes[0].legend(); axes[0].grid(alpha=.3)
h[['acc train', 'acc validation']].plot(ax=axes[1], lw=2); axes[1].set_ylabel('accuracy'); axes[1].grid(alpha=.3)
plt.tight_layout(); plt.show()

print(h.loc[[1, 10, meilleure, 150]].round(3))

> 🧐 L'accuracy d'entraînement atteint 100 % : le réseau connaît ses 200 images par cœur. Mais la **loss de validation**
> atteint son minimum assez tôt puis **remonte** : au-delà, le réseau devient de plus en plus sûr de lui sur des
> prédictions qui ne s'améliorent plus. C'est le **surapprentissage**. Le remède le plus simple :
> surveiller la validation et **arrêter tôt** (*early stopping*).

---
# PARTIE 9 — ML classique vs Deep Learning sur des données tabulaires

Le cours affirme : *sur des données tabulaires de taille modérée, le ML classique est souvent aussi bon, voire meilleur,
et plus simple*. Comparons plusieurs modèles sur un jeu tabulaire réel (**Breast Cancer** : 569 patientes, 30 mesures),
avec une **validation croisée à 5 plis**.

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier

Xb, yb = load_breast_cancer(return_X_y=True)
modeles = {
    'Régression logistique':       make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
    'Random Forest':               RandomForestClassifier(n_estimators=300, random_state=42),
    'Gradient Boosting':           HistGradientBoostingClassifier(random_state=42),
    'Réseau de neurones (64, 32)': make_pipeline(StandardScaler(),
                                                 MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=2000,
                                                               random_state=42)),
}
resultats = []
for nom, m in modeles.items():
    t0 = time.perf_counter()
    scores = cross_val_score(m, Xb, yb, cv=5, scoring='accuracy')
    resultats.append({'modèle': nom, 'accuracy moyenne': scores.mean(), 'écart-type': scores.std(),
                      'temps (s)': time.perf_counter() - t0})
pd.DataFrame(resultats).sort_values('accuracy moyenne', ascending=False).round(3)

> 🧐 Les écarts d'accuracy entre modèles sont **faibles** (de l'ordre de l'écart-type). Lors de notre exécution, c'est même
> le modèle **le plus simple**, la régression logistique, qui arrive en tête (98,1 %), devant le réseau de neurones (97,7 %),
> pour un temps de calcul dix fois plus court et des coefficients directement interprétables. Le réseau n'apporte **aucun
> avantage net** sur ce problème tabulaire. Le Gradient Boosting, souvent le meilleur sur du tabulaire, ne l'est pas ici
> non plus : d'où la règle de toujours **comparer plusieurs modèles** en validation croisée. Le Deep Learning n'est pas « meilleur » en soi :
> il brille sur les données **non structurées** (images, son, texte) et les **très grands** volumes.

---
# PARTIE 10 — Un modèle reproduit les biais de ses données

Scénario **fictif** : une entreprise entraîne un modèle de présélection de CV sur ses **décisions d'embauche passées**.
Or, historiquement, les candidats du groupe B ont été **défavorisés** à compétence égale.

In [ ]:
n = 4000
groupe_B = rng.random(n) < 0.5                                  # appartenance à un groupe (A ou B)
competence = rng.normal(0, 1, n)                                 # compétence réelle : même distribution pour A et B
# Décisions HISTORIQUES : à compétence égale, le groupe B a été pénalisé
embauche = (competence - 1.0 * groupe_B + rng.normal(0, 0.5, n)) > 0.3
# Une variable « neutre » en apparence, mais fortement liée au groupe (ex. quartier de résidence)
quartier = np.where(rng.random(n) < 0.9, groupe_B, ~groupe_B).astype(int)

print(f'Compétence moyenne  — A : {competence[~groupe_B].mean():+.2f} | B : {competence[groupe_B].mean():+.2f}')
print(f"Taux d'embauche historique — A : {embauche[~groupe_B].mean():.0%} | B : {embauche[groupe_B].mean():.0%}")

In [ ]:
def taux_par_groupe(features, nom):
    modele = LogisticRegression().fit(features, embauche)
    pred = modele.predict(features)
    return {'modèle': nom, 'taux prédit A': pred[~groupe_B].mean(), 'taux prédit B': pred[groupe_B].mean(),
            'écart A - B': pred[~groupe_B].mean() - pred[groupe_B].mean()}

pd.DataFrame([
    taux_par_groupe(np.c_[competence, groupe_B], 'compétence + groupe'),
    taux_par_groupe(np.c_[competence, quartier], 'compétence + quartier (groupe retiré)'),
    taux_par_groupe(competence.reshape(-1, 1),   'compétence seule'),
]).round(3)

> 🧐 **Trois leçons** :
> 1. Entraîné sur des décisions biaisées, le modèle **reproduit** la discrimination : à compétence égale, il recommande
>    bien moins souvent les candidats du groupe B.
> 2. **Retirer la variable sensible ne suffit pas** : le modèle retrouve l'information à travers une variable
>    « innocente » qui lui est corrélée (le quartier). On parle de **variable proxy**.
> 3. Avec la seule compétence, l'écart presque disparaît (1,5 point lors de notre exécution) : les candidats sont jugés
>    sur ce qui compte. Mais le modèle a tout de même appris sur des **étiquettes biaisées** (les décisions passées) :
>    il recommande globalement moins de candidats que ce que leur compétence justifierait. Corriger les données
>    d'entraînement est souvent nécessaire.
>
> 🔑 Le data scientist est **responsable** d'auditer les biais : mesurer les résultats **par groupe**, chercher les
> proxys, et questionner la qualité des étiquettes. *La performance ne dispense jamais de l'éthique.*

---
# ✅ Synthèse

| Partie | Ce que vous avez vérifié |
|--------|--------------------------|
| **1** | Le ML retrouve une règle (F = 1,8 C + 32) à partir d'exemples |
| **2–3** | Un neurone = somme pondérée + biais + activation ; softmax transforme des scores en probabilités |
| **4** | Sans non-linéarité, deux couches = une seule couche linéaire ; avec ReLU, le réseau sépare des cercles |
| **5** | La forward s'applique à tout un lot d'exemples en une opération matricielle |
| **6** | Le learning rate décide si la descente est lente, efficace, oscillante ou divergente |
| **7** | Les neurones cachés apprennent seuls leurs détecteurs (features) |
| **8** | Trop d'epochs sur peu de données : la loss de validation remonte (surapprentissage) |
| **9** | Sur du tabulaire, le réseau de neurones ne bat pas nettement le ML classique |
| **10** | Un modèle reproduit les biais de ses données, même sans la variable sensible |

### 🚀 Pour aller plus loin
- Partie 4 : remplacez `make_circles` par `make_moons` et testez les activations `'tanh'` et `'logistic'`.
- Partie 6 : trouvez par essais le plus grand learning rate qui converge encore (indice : la dérivée seconde vaut 2).
- Partie 8 : relancez avec `alpha=0.01` (régularisation L2) dans `MLPClassifier` : la loss de validation remonte-t-elle encore ?
- Partie 10 : calculez le taux d'embauche prédit **par tranche de compétence** et par groupe, et tracez-le.

> 🔗 Suite du module : **Checkpoint 9** (`02-...ipynb`), puis **Forward & Backward Propagation** (`03-...ipynb`).

---
*📘 Module Data Science — Fondamentaux de l'IA, du ML et du Deep Learning | Bootcamp Data Science*